# Customer Churn Analysis and Customer Intelligence

End-to-end portfolio project using SQLite, SQL, pandas, NumPy, Matplotlib, and Seaborn.

In [ ]:
import sqlite3
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
BASE=Path(".."); DB_PATH=BASE/"data"/"customer_churn.db"; OUT=BASE/"output"; CHARTS=OUT/"charts"
OUT.mkdir(exist_ok=True); CHARTS.mkdir(exist_ok=True); sns.set_theme(style="whitegrid")


## 1. Load relational tables

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    customers=pd.read_sql_query("SELECT * FROM db_customer",conn)
    subscriptions=pd.read_sql_query("SELECT * FROM db_subscription",conn)
    support=pd.read_sql_query("SELECT * FROM db_support",conn)
print(customers.shape,subscriptions.shape,support.shape)
display(customers.head())


## 2. Clean columns and data types

In [ ]:
def clean_columns(df):
    df=df.copy(); df.columns=[c.strip().lower() for c in df.columns]; return df
customers,subscriptions,support=map(clean_columns,[customers,subscriptions,support])
customers["dob"]=pd.to_datetime(customers["dob"],errors="coerce")
for c in ["subscription_start_date","renewal_date","cancellation_date"]:
    subscriptions[c]=pd.to_datetime(subscriptions[c],errors="coerce")
support["complaint_date"]=pd.to_datetime(support["complaint_date"],errors="coerce")
for c in ["monthly_charges","cltv","churn_score"]:
    subscriptions[c]=pd.to_numeric(subscriptions[c],errors="coerce")
support["escalations"]=pd.to_numeric(support["escalations"],errors="coerce").fillna(0)
support["csat_score"]=pd.to_numeric(support["csat_score"],errors="coerce")
print("Duplicate customer IDs:",customers.customerid.duplicated().sum())
display(customers.isna().sum().to_frame("missing values"))
display(subscriptions.isna().sum().to_frame("missing values"))


## 3. Aggregate support data before joining

In [ ]:
support_summary=support.groupby("customerid",as_index=False).agg(
    complaint_count=("customerid","size"), escalation_count=("escalations","sum"),
    avg_csat_score=("csat_score","mean"), latest_complaint_date=("complaint_date","max"))
analysis=customers.merge(subscriptions,on="customerid",how="left",validate="one_to_one").merge(
    support_summary,on="customerid",how="left",validate="one_to_one")
analysis["complaint_count"]=analysis["complaint_count"].fillna(0).astype(int)
analysis["escalation_count"]=analysis["escalation_count"].fillna(0)
assert len(analysis)==len(subscriptions)
print("Joined dataset:",analysis.shape)


## 4. Feature engineering

In [ ]:
AS_OF_DATE=pd.Timestamp("2024-09-30")
analysis["is_churned"]=analysis["cancellation_date"].notna()
analysis["end_date_for_tenure"]=analysis["cancellation_date"].fillna(AS_OF_DATE)
analysis["tenure_days"]=(analysis["end_date_for_tenure"]-analysis["subscription_start_date"]).dt.days
analysis["churn_month"]=analysis["cancellation_date"].dt.to_period("M").astype("string")
analysis["risk_tier"]=pd.cut(analysis["churn_score"],[-1,39,69,100],labels=["Low","Medium","High"])
analysis["revenue_at_risk"]=np.where((analysis["churn_score"]>70)&(~analysis["is_churned"]),analysis["monthly_charges"].fillna(0),0)
display(analysis[["customerid","is_churned","tenure_days","risk_tier","revenue_at_risk"]].head())


## 5. KPIs and segmentation

In [ ]:
n=len(analysis); churned=int(analysis["is_churned"].sum())
kpis=pd.Series({"Total customers":n,"Churned customers":churned,
 "Churn rate (%)":round(100*churned/n,2) if n else 0,
 "Retention rate (%)":round(100*(1-churned/n),2) if n else 0,
 "Average tenure (days)":round(analysis["tenure_days"].dropna().mean(),2),
 "Customers with complaints":int((analysis["complaint_count"]>0).sum()),
 "Customers with escalations":int((analysis["escalation_count"]>0).sum()),
 "Active high-risk customers":int(((analysis["churn_score"]>70)&(~analysis["is_churned"])).sum()),
 "Active high-risk monthly charges":round(analysis["revenue_at_risk"].sum(),2)})
display(kpis.to_frame("value")); kpis.to_json(OUT/"kpi_summary.json",indent=2)
for col in ["contract_type","plan_type","state"]:
    display(analysis.groupby(col,dropna=False).agg(customers=("customerid","nunique"),churn_rate=("is_churned","mean"))
            .assign(churn_rate_pct=lambda d:(100*d.churn_rate).round(2)).sort_values("churn_rate",ascending=False))


## 6. Visualizations

In [ ]:
plt.figure(figsize=(6,4)); sns.countplot(data=analysis,x="is_churned",hue="is_churned",legend=False)
plt.xticks([0,1],["Active / not churned","Churned"]); plt.title("Customer Churn Status"); plt.tight_layout()
plt.savefig(CHARTS/"churn_status.png",dpi=160); plt.show()
for col,name in [("contract_type","churn_by_contract.png"),("plan_type","churn_by_plan.png")]:
    rates=analysis.groupby(col,dropna=False)["is_churned"].mean().mul(100).reset_index(name="churn_rate_pct")
    plt.figure(figsize=(7,4)); sns.barplot(data=rates,x=col,y="churn_rate_pct",hue=col,legend=False)
    plt.ylabel("Churn rate (%)"); plt.title(f"Churn Rate by {col.replace('_',' ').title()}"); plt.xticks(rotation=20,ha="right")
    plt.tight_layout(); plt.savefig(CHARTS/name,dpi=160); plt.show()
plt.figure(figsize=(7,4)); sns.scatterplot(data=analysis,x="churn_score",y="monthly_charges",hue="is_churned")
plt.title("Churn Score vs Monthly Charges"); plt.tight_layout(); plt.savefig(CHARTS/"risk_vs_charges.png",dpi=160); plt.show()


## 7. Export and recommendations

In [ ]:
analysis.to_csv(OUT/"cleaned_customer_churn.csv",index=False)
print("Saved cleaned data, KPI summary, and charts in output/.")


### Business recommendations
- Prioritize high-risk active customers for retention outreach.
- Compare churn by contract and plan before changing pricing.
- Review cancellation reasons and support escalation patterns.
- Validate results on a larger, current dataset before making business-wide claims.

**Definition:** a customer is churned when `cancellation_date` is populated. Tenure for active customers uses the historical snapshot date 2024-09-30.